<a href="https://colab.research.google.com/github/lalawee/special-octo-barnacle/blob/main/CartPole_DQN_Train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Q-Network(DQN)
We train NN in experience-replay mode<br>
This is done at each step during learning <br>
Training NN done in batches<br>
##This is a simple DQN with only 1 network <br>No Target Network is used here

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import random
import gym
import numpy as np
from collections import deque # Double-Ended Queue which can add and remove elements from both ends. Supports both FIFO and LIFO.

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=

In [ ]:
#
# State output is [Pos Velocity Angle Angular-Velocity] for Cartpole
#
env = gym.make('CartPole-v1', new_step_api=True)
print(env.observation_space)
print(env.observation_space.shape)
print(env.observation_space.shape[0],'State space')
print(env.action_space.n,'Action space')

Box([-4.8000002e+00 -3.4028235e+38 -4.1887903e-01 -3.4028235e+38], [4.8000002e+00 3.4028235e+38 4.1887903e-01 3.4028235e+38], (4,), float32)
(4,)
4 State space
2 Action space


##Create Dense Neural Network (NN)

In [ ]:
#
# Note: input_dim=4 is same as input_shape=(4,)
# Input is 4 nodes (from State space - vector of 4), 2 hidden layer of 32 nodes each, output 2 nodes
#
# So, input is s, when we predict it gives us highest Q-value with its associated action
# env.observation_space.shape[0] is 4
# env.action_space.n is 2
#
model=Sequential()
model.add(Dense(32, input_dim=env.observation_space.shape[0], activation='relu')) # input is 4
model.add(Dense(32, activation='relu'))
model.add(Dense(env.action_space.n, activation='linear')) # 2 output nodes, activation is linear for regression problem
model.compile(optimizer=Adam(learning_rate=0.001, clipnorm=1.0), loss='mse') # auto clips gradients tau before updating wts

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


##Hyper Parameters

In [ ]:
GAMMA = 0.99
EPSILON = 1.0
EPSILON_MIN = 0.01
EPSILON_DECAY = 0.995
MEMORYSIZE=100000
BATCHSIZE=64

In [ ]:
memory = deque(maxlen=MEMORYSIZE) #using deque (FIFO) for experience-replay memory
score = deque(maxlen=100) #keep track of scores of consecutive 100 episodes as defined in the winning requirement of Cartpole

##Q-Learning for reference purpose

In [ ]:
#For each episode:
#    init s (reset env)
#
#    For each step till done:
#        execute eps-greedy policy to choose an action
#
#        sample next state s' and reward r based on chosen action
#
#        Q[s,a] = Q[s,a] + lr*(r + gamma*np.max(Q[s',:]) - Q[s,a]) #update Q table
#
#        s = s' # that is, we now move to state s'

##Function to implement experience-replay

In [ ]:
#
# Replay in batch mode to train NN
# (Train NN with 1 batch of random samples from replay buffer)
#
def experience_replay():
    if len(memory) < BATCHSIZE: return #only start replay after memory has at least BATCHSIZE samples

    xBatch = []
    yBatch = []

    #
    # Preparing xBatch and yBatch so as to train the NN model
    # That is, train NN to map states to QValues_sa
    #
    for s, a, r, s_, done in random.sample(memory, BATCHSIZE): # select 64 random tuples of past experience out from memory
                                                               # random.sample() returns a list of stuff
                                                               # note: s in numpy array form [[...]], cause model.predict()
                                                               #       demands this
                                                               # Note: random.sample(memory, BATCHSIZE)
                                                               #       returns a list of 64 tuples
                                                               #       each tuple has 5 items
        # Find Target
        #
        # target = r + gamma*np.max(Q[s',:])
        # note: 1. gamma*np.max(Q[s',:]) is the discounted future reward
        #       2. also, this is simple DQN, no target network used, use back only primary network
        #       3. model.predict(s_) outputs 2 Q-values based on model definition
        if not done:
            target = r + GAMMA * np.max(model.predict(s_, verbose=0))
        else:
            target = r # cause s_ is invalid for "done" cases

        # Assign target calculated to the correct Q[s,a] (there are 2 actions, one of them gives you max value)
        # model.predict(s)[0] gives you output vector of 2 Q-values for the 2 possible actions without []
        # target is scalar value
        QValues_sa = model.predict(s, verbose=0)[0] #Predict always output [[...]], we change this to [...] and assign to QValues_sa
        QValues_sa[a] = target #update it with new target value, QValues_sa is np array of shape (2,)

        # Create x,y batches of "ground truths" to train NN
        #
        # each sample in xBatch is one state s, ie, input is a vector of 4
        # with corresponding reward target (calculated ground truth)
        xBatch.append(s[0]) #s already in numpy format, change from [[...]] to [...]
                            #xBatch is the batch of states, each a numpy vector of 4 values
        yBatch.append(QValues_sa) #each QValues_sa is a vector of 2 output Q-values

    # Train NN with 1 batch of 64 samples for 1 grad descent (backprop)
    model.fit(np.array(xBatch), np.array(yBatch), epochs=1, batch_size=BATCHSIZE, verbose=0) #do the NN training

## Learning

In [ ]:
#
# SHORT CUT: OVERRIDE AND LOAD MODEL WITH PRE-TRAINED PARAMETERS
#            HELP SPEED UP TRAINING
#
from tensorflow.keras.models import load_model

# Load the model to override
model = load_model('/content/drive/MyDrive/app/SIT Robotics - RL/CartPole.keras')

In [ ]:
# ====
# MAIN
# ====

i=0 # Episode i

# if you are loading in a trained model, no need to do much exploration, exploit all the way, use eps=0.01
# otherwise if you are starting afresh in training, choose EPSILON (1.0)
#eps=EPSILON #starting EPSILON
eps=0.01

In [ ]:
import numpy as np
np.bool8=np.bool_ # temp fix for an upgrade issue in latest np, bool8 has been deprecated.

In [ ]:
##
## For each Episode
##
while True: # we continue each episode i till the winning condition is achieved
    i=i+1
    state = np.array([env.reset()]) # [[Pos Velocity Angle Angular-Velocity]], we use numpy array
                                    # we make is an array of array because of model.predict() requires it
    reward_per_episode = 0

    ##
    ## For each step inside an Episode
    ##
    for t in range(500): # each episode only has max 500 time steps in CartPole
        #
        # eps-greedy policy
        #
        if np.random.rand() <= eps:
            action = env.action_space.sample()
        else:
            action = np.argmax(model.predict(state, verbose=0)) #get indices with max value, do forward pass to predict
                                                                #the max QValues_sa of the input state, and select
                                                                #the associated action
        # step to next state
        state_, reward, done, _, _ = env.step(action) #  [...]
        next_state = np.array([state_])               # [[...]], because of model.predict() at experience_replay()

        # accumulate reward per episode
        reward_per_episode += reward

        # Add to deque memory for experience replay - To train NN later
        memory.append((state,action,reward,next_state,done)) # add into memory for experience replay as tuples

        if done:
            # temp save the weights and bias of NN
            model.save('/content/drive/MyDrive/app/SIT Robotics - RL/CartPole.keras')
            break

        state = next_state #S = S'

	      #
	      # Experience Replay at each time step to train the NN
        # Use 1 batch of experiences from memory to train
	      # NN so that it will learn to map from a state to QValues_sa (output vector of 2)
	      #
        # I.e., produce a batch of 64 values of xBatch, yBatch to fit NN inside this function with 1 epoch
        experience_replay()

    # eps-decay
    if eps > EPSILON_MIN: eps *= EPSILON_DECAY

    # record reward for this episode in deque score
    score.append(reward_per_episode)

    # print at end of each episode
    print(f"Episode: {i}, Score: {reward_per_episode}, Avg over last 100 tries: {np.mean(score):.1f}, eps={eps:0.3f}")

    # check if solved
    if np.mean(score)>=475.0: #according to rules of Cartpole, score is a FIFO deque of size 100
      print(f"Solved in {episode+1} episodes!")
      break


Episode: 2, Score: 148.0, Avg over last 100 tries: 148.0, eps=0.010
